# 08 - Quantum Teleportation

Transfer a quantum state from one qubit to another using entanglement.

**Concepts:** Bell pair, Bell measurement, classical communication

In [ ]:
import quantsdk as qs
import math

## The Protocol

1. Alice has state |psi> on qubit 0
2. Alice and Bob share a Bell pair (qubits 1-2)
3. Alice performs Bell measurement (qubits 0-1)
4. Bob applies corrections based on Alice's results

```
q0 (Alice's state):  --[Rx]--*--[H]--[M]--
q1 (Alice's Bell):   --[H]--*--[X]---[M]--
q2 (Bob's Bell):     -------X-----------[M]
```

In [ ]:
def teleportation_circuit(theta: float) -> qs.Circuit:
    """Create teleportation circuit for state RY(theta)|0>."""
    circuit = qs.Circuit(3, name=f"teleport-{theta:.2f}")
    
    # Prepare state to teleport
    circuit.ry(0, theta)
    
    # Create Bell pair (qubits 1-2)
    circuit.barrier()
    circuit.h(1).cx(1, 2)
    
    # Bell measurement
    circuit.barrier()
    circuit.cx(0, 1).h(0)
    
    # Measure all
    circuit.measure_all()
    return circuit

In [ ]:
# Teleport different states
for theta in [0, math.pi/4, math.pi/2, math.pi]:
    circuit = teleportation_circuit(theta)
    result = qs.run(circuit, shots=2000, seed=42)
    
    # Analyze Bob's qubit (q2) conditioned on Alice's results
    p_bob_1 = 0
    for bitstring, count in result.counts.items():
        if bitstring[2] == '1':  # Bob's qubit
            p_bob_1 += count
    p_bob_1 /= 2000
    
    expected = math.sin(theta / 2) ** 2
    print(f"theta={theta:.2f}: P(Bob=1)={p_bob_1:.3f} (need corrections, expected~{expected:.3f})")

In [ ]:
# Show full output for one case
circuit = teleportation_circuit(math.pi / 3)
result = qs.run(circuit, shots=4000, seed=42)

print(f"Circuit depth: {circuit.depth}")
print(f"Gate count: {circuit.gate_count}")
print(f"\nAll outcomes (q0 q1 q2):")
for bs, count in sorted(result.counts.items()):
    print(f"  |{bs}>: {count} ({count/4000:.3f})")